# Chimeric RNA-Seq Negative Data Pipeline (Updated)

This notebook implements a full pipeline to produce labeled **False Negative** and **False Positive** training data for a chimera detection model.

## 1. Terminology & Strategy

Both datasets generated here will be labeled as **False (0)**, but they serve different purposes:

* **False Negative Candidates (Baseline / Canonical):**
    * **Content:** Real, high-quality human transcripts (from UniProt Swiss-Prot).
    * **Goal:** Teach the model what "normal" RNA looks like so it doesn't flag healthy tissue.
    * **Label:** 0

* **False Positive Candidates (Hard Negatives / Synthetic):**
    * **Content:** Synthetic sequences designed to trick the model using advanced operators: **Reverse Complement**, **Shuffled**, and **Shift-Invariant Random Pairs**.
    * **Goal:** Teach the model to distinguish specific fusion breakpoints from random noise, artifacts, or non-coding strand syntax.
    * **Label:** 0

## 2. Setup & Configuration

We will use **Biopython** to handle sequence processing. Ensure it is installed (`pip install biopython`).

In [21]:
import os
import gzip
import urllib.request
import random
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# ============ CONFIGURATION ============

# Source: UniProt Swiss-Prot (Reviewed Canonical Sequences)
UNIPROT_URL = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz"

# Output Filenames
FP_OUTPUT_FILE = "false_positive_candidates.fasta"  # 'Hard' negatives (Artifacts/Synthetic)
FN_OUTPUT_FILE = "false_negative_candidates.fasta"  # 'Easy' negatives (Canonical Real RNA)

# Parameters
# Increased to 10,000 to provide better diversity for the larger 32k model
SAMPLE_SIZE = 10000 
SEED = 42

print("✅ Pipeline configured.")

✅ Pipeline configured.


## 3. Data Downloader

This step downloads the official UniProt Swiss-Prot database (gzipped) if it doesn't already exist locally.

In [22]:
def download_data(url, local_filename):
    """Downloads a file from a URL if it doesn't already exist."""
    if not os.path.exists(local_filename):
        print(f"Downloading {local_filename}...")
        try:
            urllib.request.urlretrieve(url, local_filename)
            print("Download complete.")
        except Exception as e:
            print(f"Error downloading: {e}")
            return False
    else:
        print(f"File {local_filename} already exists. Skipping download.")
    return True

# Download the UniProt fasta (gzipped)
local_gz_file = "uniprot_sprot.fasta.gz"
success = download_data(UNIPROT_URL, local_gz_file)

if not success:
    raise Exception("Failed to download necessary data.")

File uniprot_sprot.fasta.gz already exists. Skipping download.


## 4. Load Canonical Sequences

We parse the downloaded file, filtering specifically for human sequences (*Homo sapiens*) to create our "Real" baseline.

In [23]:
def load_canonical_sequences(gz_file, limit=None):
    """Parses the Gzipped FASTA file and returns a list of SeqRecords."""
    seqs = []
    print("Parsing sequences...")
    # Open the gzipped file in text mode
    with gzip.open(gz_file, "rt") as handle:
        for i, record in enumerate(SeqIO.parse(handle, "fasta")):
            # Filter for Human sequences
            if "Homo sapiens" in record.description: 
                seqs.append(record)
            
            # Stop if we reach the sample limit
            if limit and len(seqs) >= limit:
                break
    print(f"✅ Loaded {len(seqs)} canonical human sequences.")
    return seqs

# Load real biological sequences
real_transcripts = load_canonical_sequences(local_gz_file, limit=SAMPLE_SIZE)

Parsing sequences...
✅ Loaded 10000 canonical human sequences.


## 5. Mathematical Definition of Hard Negatives (Updated)

The synthetic samples (False Positive Candidates) are constructed by applying three advanced sequence operators to the canonical base set $S$, designed to prevent specific model overfitting modes (like static center bias or artifact learning).

### 5.1. Reverse Complement (Operator $\mathcal{RC}$)

Instead of simple reversal, we generate the **Reverse Complement**. This creates the opposing DNA strand sequence.
$$\mathcal{RC}(s_i) = (\mathcal{C}(b_{L}), \mathcal{C}(b_{L-1}), \dots, \mathcal{C}(b_{1}))$$
where $\mathcal{C}$ is the base-pairing function ($A \leftrightarrow T, C \leftrightarrow G$). This forces the model to learn strict 5' $\to$ 3' coding strand syntax.

---

### 5.2. Shuffled Sequence (Operator $\mathcal{P}_{\text{rand}}$)

The **random permutation operator $\mathcal{P}_{\text{rand}}$** scrambles the base order, testing if the model focuses on structural motifs over simple sequence composition.

$$\mathcal{P}_{\text{rand}}: s_i \longrightarrow s'_i$$

---

### 5.3. Shift-Invariant Junction (Operator $\mathcal{J}$)

To prevent the model from overfitting to a static central position (Lazy Learning), we use a **Stochastic Split Point** $k$.

$$\mathcal{J}(s_i, s_j, k) = s_i[1:k] \oplus s_j[k+1:L]$$

where $k \sim \mathcal{U}(0.25L, 0.75L)$. This forces the Attention Pooling head to actively scan the sequence rather than checking index 10,240.

In [24]:
# ==============================================================================
# CELL 4: GENERATE NEGATIVE DATASETS (Force Reload for Volume)
# ==============================================================================
import random
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# 1. Setup
random.seed(SEED)
TARGET_RAW_COUNT = 40000  # We need this many raw sequences to survive filtering

def trim_artifacts(seq_str):
    seq_str = seq_str.upper().rstrip('N')
    if seq_str.endswith('AAAAAAAAAA'):
        return seq_str.rstrip('A')
    return seq_str

# 2. Force Reload if Data is Insufficient
# We check if we have enough raw data in memory. If not, we fetch more.
current_count = len(real_transcripts) if 'real_transcripts' in locals() else 0

if current_count < TARGET_RAW_COUNT:
    print(f"🔄 Current pool ({current_count}) is too small. Reloading {TARGET_RAW_COUNT} sequences...")
    # Calls the function defined in Cell 3
    real_transcripts = load_canonical_sequences(local_gz_file, limit=TARGET_RAW_COUNT)
else:
    print(f"✅ Using existing pool of {current_count} sequences.")

# -------------------------------------------------------
# A. "False Negative" Dataset (Baseline / Canonical)
# -------------------------------------------------------
print(f"\nProcessing {len(real_transcripts)} base transcripts...")
fn_records = []

for record in real_transcripts:
    clean_seq = trim_artifacts(str(record.seq))
    
    # Filter: Drop sequences too short to be useful long-range negatives
    if len(clean_seq) < 500: continue
    
    new_id = f"NEG_CANONICAL_{record.id}"
    new_rec = SeqRecord(
        Seq(clean_seq),
        id=new_id,
        description="Label:0 source:UniProt_Canonical"
    )
    fn_records.append(new_rec)

print(f"  > Generated {len(fn_records)} Canonical Negatives.")

# -------------------------------------------------------
# B. "False Positive" Dataset (Synthetic / Hard Negatives)
# -------------------------------------------------------
print("\nGenerating Synthetic Hard Negatives...")
fp_records = []
strategies = ['rev_comp', 'shuffle', 'random_pair_jitter']

for record in real_transcripts:
    seq_str = trim_artifacts(str(record.seq))
    
    if len(seq_str) < 500: continue

    # AGGRESSIVE MULTIPLIER: 3 to 4 variants per valid sequence
    num_variants = random.choice([3, 4])
    chosen_strategies = random.choices(strategies, k=num_variants)
    
    for i, strat in enumerate(chosen_strategies):
        
        if strat == 'rev_comp':
            rc_seq = str(Seq(seq_str).reverse_complement())
            fp_records.append(SeqRecord(
                Seq(rc_seq),
                id=f"NEG_SYNTH_RC_{i}_{record.id}",
                description="Label:0 type:reverse_complement"
            ))
            
        elif strat == 'shuffle':
            seq_list = list(seq_str)
            random.shuffle(seq_list)
            shuffled_seq = "".join(seq_list)
            fp_records.append(SeqRecord(
                Seq(shuffled_seq),
                id=f"NEG_SYNTH_SHUFFLED_{i}_{record.id}",
                description="Label:0 type:shuffled_gc_preserved"
            ))
            
        elif strat == 'random_pair_jitter':
            other_record = random.choice(real_transcripts)
            other_seq = trim_artifacts(str(other_record.seq))
            
            if len(other_seq) < 500: continue

            split_idx_1 = random.randint(int(len(seq_str)*0.3), int(len(seq_str)*0.8))
            split_idx_2 = random.randint(int(len(other_seq)*0.2), int(len(other_seq)*0.7))
            
            hybrid_seq = seq_str[:split_idx_1] + other_seq[split_idx_2:]
            
            if len(hybrid_seq) > 500:
                fp_records.append(SeqRecord(
                    Seq(hybrid_seq),
                    id=f"NEG_SYNTH_JITTER_{i}_{record.id}_vs_{other_record.id}",
                    description="Label:0 type:shift_invariant_artifact"
                ))

# -------------------------------------------------------
# SUMMARY
# -------------------------------------------------------
total_neg = len(fn_records) + len(fp_records)
print(f"\n✅ GENERATION COMPLETE")
print(f"   ├── Canonical Negatives: {len(fn_records)}")
print(f"   ├── Synthetic Negatives: {len(fp_records)}")
print(f"   └── TOTAL NEGATIVE SET:  {total_neg} samples")

if total_neg < 30000:
    print("⚠️ Warning: Still below 30k. Try increasing TARGET_RAW_COUNT to 60000.")
else:
    print(f"🎉 Success: Created {total_neg} negative samples (Target: >30k).")

🔄 Current pool (10000) is too small. Reloading 40000 sequences...
Parsing sequences...
✅ Loaded 20423 canonical human sequences.

Processing 20423 base transcripts...
  > Generated 8080 Canonical Negatives.

Generating Synthetic Hard Negatives...

✅ GENERATION COMPLETE
   ├── Canonical Negatives: 8080
   ├── Synthetic Negatives: 22529
   └── TOTAL NEGATIVE SET:  30609 samples
🎉 Success: Created 30609 negative samples (Target: >30k).


## 7. Save to FASTA

Write the records to disk.

In [25]:
print(f"Writing to {FN_OUTPUT_FILE}...")
SeqIO.write(fn_records, FN_OUTPUT_FILE, "fasta")

print(f"Writing to {FP_OUTPUT_FILE}...")
SeqIO.write(fp_records, FP_OUTPUT_FILE, "fasta")

print("\n🎉 Processing Complete!")
print(f"1. [False Negative Candidates] -> {os.path.abspath(FN_OUTPUT_FILE)}")
print(f"2. [False Positive Candidates] -> {os.path.abspath(FP_OUTPUT_FILE)}")

Writing to false_negative_candidates.fasta...
Writing to false_positive_candidates.fasta...

🎉 Processing Complete!
1. [False Negative Candidates] -> /home/akp1/GeneAI/false_negative_candidates.fasta
2. [False Positive Candidates] -> /home/akp1/GeneAI/false_positive_candidates.fasta
